# Module 08-1: Exponential Backoff + Jitter 구현

## 학습 목표
- 지수 백오프(Exponential Backoff)의 원리를 이해하고 직접 구현할 수 있다
- Jitter(랜덤 지연)로 Thundering Herd 문제를 해결할 수 있다
- `retry_with_backoff()` 함수를 작성하고 검증할 수 있다

**참조 문서**: `docs/part4-production/08-error-handling-resilience.md` 섹션 2.1

## 개념 요약

### Exponential Backoff + Jitter

AI 에이전트는 LLM API, 외부 시스템 등 다양한 곳에서 에러가 발생합니다. 에러가 발생했을 때 단순히 즉시 재시도하면 서버에 더 큰 부하를 줍니다.

| 전략 | 대기 시간 | 문제 |
|------|----------|------|
| 즉시 재시도 | 0초 | 서버 과부하 가중 |
| 고정 대기 | 항상 2초 | 동시 재시도 → Thundering Herd |
| Exponential Backoff | 1, 2, 4, 8초... | 분산 효과 없음 |
| **Backoff + Jitter** | **0~1, 0~2, 0~4초...** | **최적: 분산 + 점진적 증가** |

### 핵심 공식

```
delay = min(base_delay * (backoff_factor ^ attempt), max_delay)
if jitter:
    delay = random.uniform(0, delay)
```

### Retriable vs Non-retriable 에러

| 에러 유형 | 재시도 여부 |
|----------|------------|
| Rate Limit (429), 서버 오류 (5xx), 타임아웃 | Retriable (재시도 가능) |
| 인증 실패 (401), 권한 없음 (403), 잘못된 요청 (400) | Non-retriable (재시도 불가) |

In [ ]:
# 환경 설정
import sys
sys.path.insert(0, '..')

import time
import random
import logging

from common.fake_llm import FakeLLM

logging.basicConfig(level=logging.WARNING)
logger = logging.getLogger(__name__)

print("환경 설정 완료")

## 실습 1: retry_with_backoff() 함수 구현

`retry_with_backoff()` 함수를 구현합니다.

이 함수는:
- 주어진 함수 `fn`을 최대 `max_retries`번 재시도합니다
- 재시도 간격은 지수적으로 증가합니다 (`base_delay * backoff_factor^attempt`)
- `max_delay`를 초과하지 않도록 제한합니다
- `jitter=True`이면 `random.uniform(0, delay)`로 랜덤성을 추가합니다

In [ ]:
# TODO: 여기에 코드를 작성하세요
#
# 힌트 1 (방향): for attempt in range(max_retries + 1) 루프를 사용하세요.
#               try/except로 retriable_exceptions를 잡고, attempt < max_retries이면
#               delay를 계산하여 time.sleep() 합니다.
#
# 힌트 2 (키워드): range(max_retries + 1), retriable_exceptions, backoff_factor ** attempt,
#               min(delay, max_delay), random.uniform(0, delay), time.sleep(delay)
#
# 힌트 3 (거의 정답):
#   def retry_with_backoff(fn, max_retries=3, base_delay=1.0, max_delay=30.0,
#                          backoff_factor=2.0, jitter=True, retriable_exceptions=(Exception,)):
#       last_exception = None
#       for attempt in range(max_retries + 1):
#           try:
#               return fn()
#           except retriable_exceptions as exc:
#               last_exception = exc
#               if attempt >= max_retries:
#                   raise
#               delay = min(base_delay * (backoff_factor ** attempt), max_delay)
#               if jitter:
#                   delay = random.uniform(0, delay)
#               print(f"재시도 {attempt + 1}/{max_retries}, {delay:.2f}초 대기...")
#               time.sleep(delay)
#       raise last_exception

def retry_with_backoff(
    fn,
    max_retries: int = 3,
    base_delay: float = 1.0,
    max_delay: float = 30.0,
    backoff_factor: float = 2.0,
    jitter: bool = True,
    retriable_exceptions: tuple = (Exception,),
):
    """Exponential Backoff + Jitter로 함수를 재시도합니다.

    Args:
        fn: 실행할 함수 (인자 없는 callable)
        max_retries: 최대 재시도 횟수
        base_delay: 첫 번째 재시도 전 기본 대기 시간(초)
        max_delay: 최대 대기 시간(초)
        backoff_factor: 대기 시간 증가 배수 (보통 2.0)
        jitter: True면 랜덤 지연 추가
        retriable_exceptions: 재시도할 예외 타입

    Returns:
        fn()의 반환값

    Raises:
        마지막 시도에서 발생한 예외
    """
    last_exception = None

    # TODO: max_retries + 1번 시도하는 루프를 구현하세요
    # TODO: retriable_exceptions를 잡아 delay 계산 후 sleep
    # TODO: 마지막 시도 실패 시 예외를 raise

    pass  # 이 줄을 지우고 구현하세요


print("retry_with_backoff 함수 정의 완료")

In [ ]:
# 검증 셀: 성공 케이스
call_count = 0

def succeed_on_third():
    global call_count
    call_count += 1
    if call_count < 3:
        raise ConnectionError(f"연결 실패 (시도 {call_count}회)")
    return "성공!"

call_count = 0
result = retry_with_backoff(
    fn=succeed_on_third,
    max_retries=3,
    base_delay=0.01,  # 테스트용으로 짧게
    jitter=False,
    retriable_exceptions=(ConnectionError,),
)

assert result == "성공!", f"result가 '성공!'이어야 합니다. 현재: {result}"
assert call_count == 3, f"3번 시도해야 합니다. 현재: {call_count}번"
print(f"성공 케이스 통과: {call_count}번 시도 후 성공")

In [ ]:
# 검증 셀: 실패 케이스 (모든 재시도 소진)
fail_count = 0

def always_fail():
    global fail_count
    fail_count += 1
    raise ValueError("항상 실패")

fail_count = 0
try:
    retry_with_backoff(
        fn=always_fail,
        max_retries=2,
        base_delay=0.01,
        jitter=False,
    )
    assert False, "예외가 발생해야 합니다"
except ValueError as e:
    pass

assert fail_count == 3, f"초기 1회 + 재시도 2회 = 총 3회여야 합니다. 현재: {fail_count}회"
print(f"실패 케이스 통과: {fail_count}번 시도 후 예외 발생 확인")
print("실습 1 완료! retry_with_backoff() 구현 성공")

## 실습 2: max_delay 제한 + Jitter 동작 확인

대기 시간이 `max_delay`를 초과하지 않는지,  
그리고 `jitter=True`일 때 대기 시간에 랜덤성이 추가되는지 확인합니다.

In [ ]:
# TODO: 여기에 코드를 작성하세요
#
# 힌트 1 (방향): time.sleep을 모의(mock)로 대체하여 실제 대기 없이 테스트합니다.
#               sleep에 전달된 값을 기록하여 max_delay 초과 여부를 검증하세요.
#
# 힌트 2 (키워드): unittest.mock.patch, sleep_durations.append, all(d <= max_delay for d in ...)
#
# 힌트 3 (거의 정답):
#   from unittest.mock import patch
#   sleep_durations = []
#   with patch('time.sleep', side_effect=lambda d: sleep_durations.append(d)):
#       try:
#           retry_with_backoff(fn=lambda: (_ for _ in ()).throw(RuntimeError("fail")),
#                              max_retries=4, base_delay=1.0, max_delay=3.0,
#                              backoff_factor=2.0, jitter=False)
#       except RuntimeError:
#           pass

from unittest.mock import patch

sleep_durations = []
MAX_DELAY = 3.0

# TODO: time.sleep을 패치하여 sleep_durations에 기록하세요
# TODO: retry_with_backoff를 max_retries=4, base_delay=1.0, max_delay=MAX_DELAY, jitter=False로 호출

print(f"sleep 호출 대기 시간들: {[round(d, 2) for d in sleep_durations]}")

In [ ]:
# 검증 셀
assert len(sleep_durations) == 4, f"4번 재시도해야 sleep이 4번 호출됩니다. 현재: {len(sleep_durations)}번"
assert all(d <= MAX_DELAY for d in sleep_durations), (
    f"모든 대기 시간이 max_delay({MAX_DELAY})를 초과하지 않아야 합니다. "
    f"현재: {sleep_durations}"
)
# jitter=False이므로 결정론적으로 계산 가능
expected = [min(1.0 * (2.0 ** i), MAX_DELAY) for i in range(4)]
for actual, exp in zip(sleep_durations, expected):
    assert abs(actual - exp) < 0.001, f"예상값 {exp}와 실제값 {actual}이 다릅니다"

print(f"max_delay 제한 통과: 모든 대기 시간 <= {MAX_DELAY}초")
print(f"예상 대기 시간: {[round(d,2) for d in expected]}")
print(f"실제 대기 시간: {[round(d,2) for d in sleep_durations]}")
print("실습 2 완료!")

## 실습 3: FakeLLM과 연동하여 재시도 흐름 시뮬레이션

FakeLLM을 활용하여 LLM 호출이 일시적으로 실패했다가 성공하는 시나리오를 시뮬레이션합니다.

In [ ]:
# TODO: 여기에 코드를 작성하세요
#
# 힌트 1 (방향): 카운터를 사용하여 처음 2회는 예외를 발생시키고,
#               3번째 호출부터 FakeLLM 응답을 반환하는 함수를 만드세요.
#
# 힌트 2 (키워드): attempt_count, ConnectionError("Rate limit"), FakeLLM().invoke()
#
# 힌트 3 (거의 정답):
#   attempt_count = 0
#   llm = FakeLLM(responses={"분석": "분석 완료: 버그 없음"})
#   def flaky_llm_call():
#       global attempt_count
#       attempt_count += 1
#       if attempt_count <= 2:
#           raise ConnectionError(f"Rate limit exceeded (시도 {attempt_count}회)")
#       return llm.invoke("코드를 분석해주세요").content

attempt_count = 0
llm = FakeLLM(responses={"분석": "분석 완료: 버그 없음"})

def flaky_llm_call():
    global attempt_count
    attempt_count += 1
    # TODO: 처음 2회는 ConnectionError, 이후에는 LLM 응답 반환
    pass  # 이 줄을 지우고 구현하세요

attempt_count = 0
result = retry_with_backoff(
    fn=flaky_llm_call,
    max_retries=3,
    base_delay=0.01,  # 테스트용 짧은 대기
    jitter=False,
    retriable_exceptions=(ConnectionError,),
)

print(f"최종 결과: {result}")
print(f"총 시도 횟수: {attempt_count}")

In [ ]:
# 검증 셀
assert attempt_count == 3, f"총 3번 시도해야 합니다 (2번 실패 + 1번 성공). 현재: {attempt_count}번"
assert result is not None, "result가 None입니다"
assert "분석" in result, f"FakeLLM 응답에 '분석'이 포함되어야 합니다. 현재: {result}"
print(f"FakeLLM 연동 테스트 통과: {attempt_count}번 시도 후 성공")
print(f"응답: {result}")
print("실습 3 완료!")

## 정리

이번 실습에서 학습한 내용:

1. **Exponential Backoff**: 재시도 간격을 지수적으로 증가시켜 서버 부하를 줄임
2. **Jitter**: `random.uniform(0, delay)`로 재시도 시점을 분산시켜 Thundering Herd 방지
3. **max_delay 제한**: 무한정 대기하지 않도록 상한선 설정
4. **Retriable 에러 선별**: `retriable_exceptions`로 재시도할 예외만 지정

**핵심 공식**: `delay = min(base_delay * (backoff_factor^attempt), max_delay)`

**다음 실습**: `02_circuit_breaker.ipynb` - Circuit Breaker 패턴으로 반복 실패 차단